# Eksperyment: Walidacja walk-forward zestawów cech (nowe dane 2020-2025)

## Cel
Domknąć luki dowodowe: na nowych danych (sezony 2020-2025) sprawdzić, czy którykolwiek z
czterech kandydujących zestawów cech przebija baseline w sposób istotny statystycznie. Testujemy
cztery zestawy doklejane do baseline:

| zestaw | cech | co wnosi |
|---|---|---|
| `surface`  | 3 | `court_pace_index` + 2 interakcje serwis x prędkość kortu |
| `fatigue`  | 6 | `rest_days` + `tourney_minutes` (dni odpoczynku + minuty w turnieju) |
| `enriched` | 9 | `surface` + `fatigue` razem |
| `elo`      | 4 | `elo_diff`, `surface_elo_diff`, `elo_win_prob`, `surface_elo_win_prob` |

## Metoda (leakage-safe, czysta ablacja)
- **Walk-forward** przez 6 sezonów (2020-2025): dla każdego roku osobno trenujemy baseline na
  jego własnych meczach (te same chronologiczne splity 60/20/20), a potem doklejamy zestaw cech.
- Cechy liczone z historii właściwej dla danego roku (`2001..rok-1` dla Elo, `HISTORY_START_YEAR`
  dla surface/fatigue) -- model nigdy nie widzi przyszłości.
- Te same tuned hiperparametry co baseline (`search.best_params_`) -- zmieniamy *wyłącznie* cechy.
- Parujemy wynik mecz-po-meczu (kto trafił) -> pooled delta + parowany test **McNemara**, bo
  pojedynczy sezon potrafi skłamać.

In [ ]:
import sys
from pathlib import Path
import os

_REPO = "https://github.com/StanislawKarwala/TennisPredictionModel.git"
if "google.colab" in sys.modules and not Path("../src/tennis_model.py").exists():
    import subprocess
    subprocess.run(["pip", "install", "-q", "xgboost"])
    subprocess.run(["git", "clone", "-q", _REPO, "/content/tenis"])
    os.chdir("/content/tenis/notebooks")
sys.path.insert(0, str(Path("../src").resolve()))

## 1. Reuse modułu walidacyjnego
Importujemy gotowe funkcje z `tennis_model_validate_features.py` -- nie duplikujemy logiki, tylko
narratujemy wokół niej. Kluczowe elementy:
- `run_baseline(year)` -- odpala (cicho, z cache) `tennis_model.py` dla danego roku i zwraca jego
  namespace (splity, tuned HP, funkcje pomocnicze, wynik baseline).
- `build_context(ns, year)` -- liczy per-match kontekst (court pace + fatigue + Elo) wyrównany do
  splitów baseline.
- `eval_featureset(ns, {"ctx": ctx}, feature_list)` -- trenuje RF na `baseline + feature_list`
  i zwraca (winner-perspective `correct_prediction`, match accuracy).
- `mcnemar(b, c)` -- parowany test istotności.

In [ ]:
import os
import numpy as np
import pandas as pd

from tennis_model_validate_features import (
    run_baseline, build_context, eval_featureset, mcnemar,
    TARGET_YEARS, SURFACE_FEATURES, FATIGUE_FEATURES, ELO_FEATS,
    HISTORY_START_YEAR,
)

SETS = {
    "surface":  SURFACE_FEATURES,
    "fatigue":  FATIGUE_FEATURES,
    "enriched": SURFACE_FEATURES + FATIGUE_FEATURES,
    "elo":      ELO_FEATS,
}

print(f"Sezony walk-forward: {TARGET_YEARS}")
print(f"Historia cech zaczyna się w: {HISTORY_START_YEAR}")
print("\nZestawy cech testowane na każdym roku:")
for name, feats in SETS.items():
    print(f"  {name:<10} ({len(feats)} cech): {feats}")

Sezony walk-forward: [2020, 2021, 2022, 2023, 2024, 2025]
Historia cech zaczyna się w: 2001

Zestawy cech testowane na każdym roku:
  surface    (3 cech): ['court_pace_index', 'ace_speed_diff', 'first_won_speed_diff']
  fatigue    (6 cech): ['p1_rest_days', 'p2_rest_days', 'rest_days_diff', 'p1_tourney_minutes', 'p2_tourney_minutes', 'tourney_minutes_diff']
  enriched   (9 cech): ['court_pace_index', 'ace_speed_diff', 'first_won_speed_diff', 'p1_rest_days', 'p2_rest_days', 'rest_days_diff', 'p1_tourney_minutes', 'p2_tourney_minutes', 'tourney_minutes_diff']
  elo        (4 cech): ['elo_diff', 'surface_elo_diff', 'elo_win_prob', 'surface_elo_win_prob']


## 2. Narracyjne demo: jeden sezon (2025)
Zanim odpalimy pełną pętlę 6 sezonów, rozbierzmy jeden rok na czynniki pierwsze. Bierzemy 2025:
liczymy baseline, budujemy kontekst (court pace + fatigue + Elo), pokazujemy próbkę kontekstu, a potem
ewaluujemy każdy zestaw cech i porównujemy match accuracy z baseline.

In [ ]:
DEMO_YEAR = 2025
ns_demo = run_baseline(DEMO_YEAR)                       # runpy tennis_model.py (cicho, cache)
base_eval_demo = ns_demo["winner_perspective"][["match_id", "correct_prediction"]].copy()
base_match_demo = float(ns_demo["match_accuracy"])

n_tr = len(ns_demo["df_train_raw"]); n_va = len(ns_demo["df_val_raw"]); n_te = len(ns_demo["df_test_raw"])
print(f"Baseline {DEMO_YEAR}: match={base_match_demo:.4f}  |  split train/val/test = {n_tr}/{n_va}/{n_te}")
print(f"Cech baseline: {len(ns_demo['features'])}   |   tuned HP: {ns_demo['search'].best_params_}")

Baseline 2025: match=0.6566  |  split train/val/test = 1588/529/530
Cech baseline: 40   |   tuned HP: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 8, 'max_samples': 0.9, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': True}


### 2a. Budujemy kontekst (court pace + fatigue + Elo)
`build_context` zwraca `(full_base, ctx)`, gdzie `ctx` ma jeden wiersz per mecz wyrównany pozycyjnie
do splitów baseline: `court_pace_index`, `w_/l_rest_days`, `w_/l_tourney_minutes` oraz cztery kolumny
Elo (`w_elo`, `l_elo`, `w_surface_elo`, `l_surface_elo`). Pokażmy próbkę i podstawowe statystyki.

In [ ]:
full_base_demo, ctx_demo = build_context(ns_demo, DEMO_YEAR)
assert len(ctx_demo) == n_tr + n_va + n_te, "Niespójność długości kontekstu vs baseline"

print(f"Kontekst: {len(ctx_demo)} wierszy  |  kolumny: {list(ctx_demo.columns)}")
print("\nPróbka (pierwsze 5 meczów):")
print(ctx_demo.head().to_string(index=False, float_format=lambda x: f"{x:.2f}"))

print("\nStatystyki wybranych kolumn kontekstu:")
print(ctx_demo[["court_pace_index", "w_rest_days", "w_tourney_minutes", "w_elo"]]
      .describe().loc[["mean", "std", "min", "max"]]
      .to_string(float_format=lambda x: f"{x:.2f}"))

Kontekst: 2647 wierszy  |  kolumny: ['court_pace_index', 'w_rest_days', 'l_rest_days', 'w_tourney_minutes', 'l_tourney_minutes', 'w_elo', 'l_elo', 'w_surface_elo', 'l_surface_elo']

Próbka (pierwsze 5 meczów):
 court_pace_index  w_rest_days  l_rest_days  w_tourney_minutes  l_tourney_minutes   w_elo   l_elo  w_surface_elo  l_surface_elo
             0.68        60.00        54.00               0.00               0.00 1544.06 1806.57        1528.38        1730.01
             0.68         0.00        60.00              76.00               0.00 1778.36 1664.71        1693.24        1549.95
             0.68         0.00         0.00              76.00             164.00 1619.86 1638.91        1601.33        1519.62
             0.68        46.00        54.00               0.00               0.00 2126.28 1728.27        2072.07        1599.59
             0.68        60.00         0.00               0.00              77.00 1764.09 1725.65        1635.38        1596.98

Statystyki wybranych 

### 2b. Ewaluacja każdego zestawu cech dla 2025
Dla każdego zestawu trenujemy RF z **tymi samymi** HP co baseline (różnica = tylko dodane cechy) i
porównujemy match accuracy. To pojedynczy sezon -- pokaże kierunek, ale o istotności zdecyduje dopiero
pełna pętla walk-forward.

In [ ]:
print(f"baseline {DEMO_YEAR}: match={base_match_demo:.4f}\n")
for name, feats in SETS.items():
    ev, match = eval_featureset(ns_demo, {"ctx": ctx_demo}, feats)
    merged = base_eval_demo.merge(ev, on="match_id", suffixes=("_base", "_var"))
    b = int(np.sum(merged["correct_prediction_base"] & ~merged["correct_prediction_var"]))
    c = int(np.sum(~merged["correct_prediction_base"] & merged["correct_prediction_var"]))
    print(f"  {name:<10} match={match:.4f}  delta={match - base_match_demo:+.4f}  "
          f"(zmienione: baseline-only b={b}, wariant-only c={c})")

baseline 2025: match=0.6566



  surface    match=0.6566  delta=+0.0000  (zmienione: baseline-only b=7, wariant-only c=7)


  fatigue    match=0.6566  delta=+0.0000  (zmienione: baseline-only b=8, wariant-only c=8)


  enriched   match=0.6585  delta=+0.0019  (zmienione: baseline-only b=6, wariant-only c=7)


  elo        match=0.6377  delta=-0.0189  (zmienione: baseline-only b=23, wariant-only c=13)


## 3. Pełna walidacja walk-forward 2020-2025
Teraz uczciwy bieg przez wszystkie sezony. Dla **każdego** roku:
1. liczymy baseline (`run_baseline`) i jego per-mecz trafienia,
2. budujemy kontekst (`build_context`),
3. dla każdego z czterech zestawów trenujemy `baseline + zestaw` i parujemy mecz-po-meczu z baseline.

To długi bieg -- 6x pełny baseline + 4 modele per rok. Zbieramy per-rok delty oraz globalne pary
(baseline trafił / wariant trafił) do późniejszego McNemara.

In [ ]:
pairs = {k: [] for k in SETS}
per_year = {k: [] for k in SETS}

for year in TARGET_YEARS:
    print(f"\n===== ROK {year} =====", flush=True)
    ns = run_baseline(year)
    base_eval = ns["winner_perspective"][["match_id", "correct_prediction"]].copy()
    base_match = float(ns["match_accuracy"])
    print(f"  baseline match={base_match:.4f}", flush=True)
    _, ctx = build_context(ns, year)
    for name, feats in SETS.items():
        ev, match = eval_featureset(ns, {"ctx": ctx}, feats)
        merged = base_eval.merge(ev, on="match_id", suffixes=("_base", "_var"))
        for _, r in merged.iterrows():
            pairs[name].append((bool(r["correct_prediction_base"]), bool(r["correct_prediction_var"])))
        per_year[name].append({"year": year, "baseline": base_match,
                               "variant": match, "delta": match - base_match})
        print(f"    {name:<10} match={match:.4f}  delta={match - base_match:+.4f}", flush=True)

os.environ.pop("TENNIS_TARGET_YEAR", None)
print("\nWalk-forward zakończony.")


===== ROK 2020 =====
  baseline match=0.6176
    surface    match=0.6250  delta=+0.0074
    fatigue    match=0.6140  delta=-0.0037
    enriched   match=0.6140  delta=-0.0037
    elo        match=0.6434  delta=+0.0257

===== ROK 2021 =====
  baseline match=0.6667
    surface    match=0.6762  delta=+0.0096
    fatigue    match=0.6609  delta=-0.0057
    enriched   match=0.6667  delta=+0.0000
    elo        match=0.6762  delta=+0.0096

===== ROK 2022 =====
  baseline match=0.6728
    surface    match=0.6782  delta=+0.0055
    fatigue    match=0.6746  delta=+0.0018
    enriched   match=0.6764  delta=+0.0037
    elo        match=0.6947  delta=+0.0219

===== ROK 2023 =====


## 4. Tabele per-rok + pooled delta + McNemar
Dla każdego zestawu składamy tabele sezon-po-sezonie, liczymy **pooled** match accuracy na wszystkich
sparowanych meczach i parowany test McNemara. `b` = mecze, które trafił tylko baseline; `c` = mecze,
które trafił tylko wariant. Istotność na korzyść cechy wymaga `p<0.05` przy `c>b`.

In [ ]:
summary = []
for name in SETS:
    df = pd.DataFrame(per_year[name])
    arr = np.array(pairs[name]); bc, vc = arr[:, 0], arr[:, 1]
    b = int(np.sum(bc & ~vc)); c = int(np.sum(~bc & vc))
    z, p = mcnemar(b, c)
    pos = int((df["delta"] > 0).sum())
    sig = ("ISTOTNE" if (p < 0.05 and c > b)
           else ("ISTOTNE-na-niekorzyść" if p < 0.05 else "brak istotności"))
    print("\n" + "=" * 74)
    print(f"--- {name} ({len(SETS[name])} cech) ---")
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"  POOLED ({len(arr)}): baseline={bc.mean():.4f}  {name}={vc.mean():.4f}  "
          f"delta={vc.mean() - bc.mean():+.4f}  (dodatnie {pos}/{len(df)} lat)")
    print(f"  McNemar: b={b} c={c} z={z:.2f} p={p:.4f} -> {sig}")
    summary.append({"zestaw": name, "cech": len(SETS[name]),
                    "baseline": bc.mean(), "wariant": vc.mean(),
                    "delta": vc.mean() - bc.mean(), "p": p, "ocena": sig})

print("\n" + "=" * 74)
print("PODSUMOWANIE ZBIORCZE (pooled 2020-2025)")
print("=" * 74)
print(pd.DataFrame(summary).to_string(index=False, float_format=lambda x: f"{x:.4f}"))


--- surface (3 cech) ---
 year  baseline  variant  delta
 2020    0.6176   0.6250 0.0074
 2021    0.6667   0.6762 0.0096
 2022    0.6728   0.6782 0.0055
 2023    0.6346   0.6435 0.0089
 2024    0.6186   0.6237 0.0051
 2025    0.6566   0.6566 0.0000
  POOLED (3022): baseline=0.6463  surface=0.6522  delta=+0.0060  (dodatnie 5/6 lat)
  McNemar: b=46 c=64 z=1.62 p=0.1050 -> brak istotności

--- fatigue (6 cech) ---
 year  baseline  variant   delta
 2020    0.6176   0.6140 -0.0037
 2021    0.6667   0.6609 -0.0057
 2022    0.6728   0.6746  0.0018
 2023    0.6346   0.6381  0.0036
 2024    0.6186   0.6220  0.0034
 2025    0.6566   0.6566  0.0000
  POOLED (3022): baseline=0.6463  fatigue=0.6466  delta=+0.0003  (dodatnie 3/6 lat)
  McNemar: b=54 c=55 z=0.00 p=1.0000 -> brak istotności

--- enriched (9 cech) ---
 year  baseline  variant   delta
 2020    0.6176   0.6140 -0.0037
 2021    0.6667   0.6667  0.0000
 2022    0.6728   0.6764  0.0037
 2023    0.6346   0.6346  0.0000
 2024    0.6186   0.6

## Wnioski
Walidacja przez 6 sezonów (2020–2025, ~3000 meczów, baseline 0,6463) daje spójny obraz — żaden zestaw cech nie przebija baseline w sposób istotny:

| zestaw cech | poprawa | McNemar p |
|---|---|---|
| prędkość kortu | +0,60 p.p. | 0,105 |
| zmęczenie | +0,03 p.p. | 1,000 |
| prędkość + zmęczenie | +0,20 p.p. | 0,656 |
| Elo | +0,76 p.p. | 0,173 |

Wszystkie poprawy są małe i nieistotne (p > 0,05). Powód jest prosty: ranking ATP i forma, które model już ma, mieszczą w sobie większość tego, co te cechy próbują wnieść — więc dokładanie ich powtarza istniejący sygnał, zamiast dodać nowy. To znów ten sam sufit ~65%.